# Plots FedMAD (adaptado de plots.ipynb)

Adaptacao do notebook original do PFLlib para os resultados do sistema MAD.
Estrutura identica: metricas (acc/auc/loss), deteccao (FPR/FRR/removidos), comparacao entre experimentos e agregacao por cc.

**Fonte de dados:** `*_agentlog.json` (escrito por `servermad.save_agent_results()`).
O `train_loss` so existe no `.h5` correspondente (PFLlib `save_results()`); sem ele, o painel de loss mostra aviso.

In [ ]:
import json
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = 'results/'
PLOTS_DIR = os.path.join(RESULTS_DIR, 'plots')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

In [ ]:
agentlog_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*_agentlog.json')))
h5_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.h5')))

print(f'Arquivos agentlog (FedMAD) ({len(agentlog_files)}):')
for f in agentlog_files:
    print(f'  {os.path.basename(f)}')

print(f'\nArquivos H5 (PFLlib, para train_loss) ({len(h5_files)}):')
for f in h5_files:
    print(f'  {os.path.basename(f)}')

In [ ]:
def parse_config(stem):
    """Configuracao extraida do nome: dataset_algo_cc_rfake_nmal_goal_times."""
    parts = stem.split('_')
    return {
        'dataset': parts[0],
        'algorithm': parts[1],
        'cc': parts[2],
        'rate_fake': parts[3] if len(parts) > 3 else '?',
        'nmal': parts[4] if len(parts) > 4 else '?',
        'goal_times': '_'.join(parts[5:]),
    }


def load_agentlog(path):
    with open(path) as f:
        data = json.load(f)
    data['_file'] = os.path.basename(path)
    stem = data['_file'][:-len('_agentlog.json')]
    data['_config'] = parse_config(stem)

    # train_loss nao esta no agentlog: tenta casar com o .h5 correspondente
    data['rs_train_loss'] = np.array([])
    h5 = os.path.join(os.path.dirname(path) or '.', stem + '.h5')
    if os.path.exists(h5):
        try:
            import h5py
            with h5py.File(h5, 'r') as hf:
                if 'rs_train_loss' in hf:
                    data['rs_train_loss'] = np.array(hf['rs_train_loss'])
        except Exception:
            pass
    return data


def detection_metrics(data):
    """FPR/FRR, removidos, TP/FN por round a partir do ground truth."""
    mali = set(data['malicious_indices'])
    n_mal = len(mali)
    n_ben = data['num_clients'] - n_mal
    rounds, removed, fpr, frr, tp, fn = [], [], [], [], [], []
    for e in data['agent_round_log']:
        rem = set(e['removed_ids'])
        t = len(rem & mali)
        f = len(rem - mali)
        n = n_mal - t if n_mal else 0
        rounds.append(e['round'])
        removed.append(len(rem))
        fpr.append(f / n_ben if n_ben else 0.0)
        frr.append(n / n_mal if n_mal else 0.0)
        tp.append(t)
        fn.append(n)
    return (np.array(rounds), np.array(removed),
            np.array(fpr), np.array(frr), np.array(tp), np.array(fn))


def moving_average(x, w):
    """Media movel igual a pandas .rolling(w).mean() (NaN no inicio)."""
    if w <= 1 or len(x) == 0:
        return x.copy()
    c = np.cumsum(np.insert(x.astype(float), 0, 0.0))
    out = (c[w:] - c[:-w]) / w
    return np.concatenate([np.full(w - 1, np.nan), out])


def detection_f1(tp, fn, removed):
    prec = np.where(removed > 0, tp / np.maximum(removed, 1), 0.0)
    den = tp + fn
    rec = np.where(den > 0, tp / np.maximum(den, 1), 0.0)
    return np.where((prec + rec) > 0, 2 * prec * rec / (prec + rec), 0.0)

In [ ]:
AGENT_INDEX = -1

if agentlog_files:
    data = load_agentlog(agentlog_files[AGENT_INDEX])
    cfg = data['_config']
    print(f'Arquivo: {data["_file"]}\n')

    print('--- Configuracao do Experimento ---')
    for k, v in cfg.items():
        print(f'  {k}: {v}')
    print(f'  clientes: {data["num_clients"]}')
    print(f'  rounds globais: {data["global_rounds"]}')
    print(f'  maliciosos: {data["n_client_malicious"]} -> {data["malicious_indices"]}')
    print(f'  agentes: {", ".join(data["agent_names"])}')

    print('\n--- Desempenho (melhores) ---')
    if len(data['rs_test_acc']):
        print(f'  test_acc best: {max(data["rs_test_acc"]):.4f}')
    if len(data['rs_test_auc']):
        print(f'  test_auc best: {max(data["rs_test_auc"]):.4f}')

    print('\n--- Deteccao por round ---')
    rounds, removed, fpr, frr, tp, fn = detection_metrics(data)
    for r, nrem, t in zip(rounds, removed, tp):
        print(f'  round {int(r):3d}: removidos={int(nrem):2d}  acertou_malicioso={int(t)}/{len(set(data["malicious_indices"]))}')
else:
    print('Nenhum arquivo agentlog encontrado.')

In [ ]:
if agentlog_files:
    data = load_agentlog(agentlog_files[AGENT_INDEX])
    cfg = data['_config']
    test_acc = np.array(data['rs_test_acc'])
    test_auc = np.array(data['rs_test_auc'])
    train_loss = np.array(data['rs_train_loss'])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    title = f'{cfg["algorithm"]} | cc={cfg["cc"]} | nmal={cfg["nmal"]}'

    # Accuracy
    if len(test_acc) > 0:
        axes[0].plot(test_acc, color='#2196F3', linewidth=1.5)
        axes[0].set_title('Test Accuracy')
        axes[0].set_xlabel('Evaluation Round')
        axes[0].set_ylabel('Accuracy')
        best = test_acc.max()
        axes[0].axhline(y=best, color='red', linestyle='--', alpha=0.5, label=f'Best: {best:.4f}')
        axes[0].legend()

    # AUC
    if len(test_auc) > 0:
        axes[1].plot(test_auc, color='#4CAF50', linewidth=1.5)
        axes[1].set_title('Test AUC')
        axes[1].set_xlabel('Evaluation Round')
        axes[1].set_ylabel('AUC')
        best_auc = test_auc.max()
        axes[1].axhline(y=best_auc, color='red', linestyle='--', alpha=0.5, label=f'Best: {best_auc:.4f}')
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, 'AUC nao disponivel', ha='center', va='center', transform=axes[1].transAxes)

    # Loss (requer o .h5)
    if len(train_loss) > 0:
        axes[2].plot(train_loss, color='#FF5722', linewidth=1.5)
        axes[2].set_title('Train Loss')
        axes[2].set_xlabel('Evaluation Round')
        axes[2].set_ylabel('Loss')
        min_loss = train_loss.min()
        axes[2].axhline(y=min_loss, color='blue', linestyle='--', alpha=0.5, label=f'Min: {min_loss:.4f}')
        axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, 'Train Loss nao disponivel\n(requer o .h5)', ha='center', va='center', transform=axes[2].transAxes)

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Nenhum arquivo agentlog encontrado.')

In [ ]:
if agentlog_files:
    data = load_agentlog(agentlog_files[AGENT_INDEX])
    rounds, removed, fpr, frr, tp, fn = detection_metrics(data)
    print(f'Arquivo: {data["_file"]}')
    print(f'Total de rounds de deteccao: {len(rounds)}')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(rounds, fpr, color='#E91E63', linewidth=1.5, label='FPR')
    axes[0].plot(rounds, frr, color='#9C27B0', linewidth=1.5, label='FRR')
    axes[0].set_title('FPR e FRR por Rodada')
    axes[0].set_xlabel('Round')
    axes[0].legend()

    axes[1].plot(rounds, removed, color='#FF9800', linewidth=1.5)
    axes[1].set_title('Clientes Removidos por Rodada')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('Nº Removidos')

    # Media movel de FPR e FRR
    window = min(10, len(rounds))
    if window > 1:
        fpr_ma = moving_average(fpr, window)
        frr_ma = moving_average(frr, window)
        axes[2].plot(rounds, fpr_ma, color='#E91E63', linewidth=1.5, label=f'FPR (media {window}r)')
        axes[2].plot(rounds, frr_ma, color='#9C27B0', linewidth=1.5, label=f'FRR (media {window}r)')
        axes[2].set_title(f'FPR/FRR Media Movel ({window} rounds)')
        axes[2].set_xlabel('Round')
        axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, 'Poucos dados para media movel', ha='center', va='center', transform=axes[2].transAxes)

    plt.suptitle('Deteccao MAD - FPR/FRR', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Nenhum arquivo agentlog encontrado.')

In [ ]:
# Selecione os indices dos arquivos agentlog para comparar
# Exemplo: COMPARE_INDICES = [0, 1, 2] para comparar os 3 primeiros
COMPARE_INDICES = list(range(len(agentlog_files)))  # todos por padrao

if len(agentlog_files) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for idx in COMPARE_INDICES:
        if idx >= len(agentlog_files):
            continue
        d = load_agentlog(agentlog_files[idx])
        cfg = d['_config']
        acc = np.array(d['rs_test_acc'])
        rounds, removed, fpr, frr, tp, fn = detection_metrics(d)
        label = f'{cfg["algorithm"]}_cc{cfg["cc"]}_nmal{cfg["nmal"]}'

        if len(acc) > 0:
            axes[0].plot(acc, linewidth=1.5, label=label)
        if len(rounds) > 0:
            f1 = detection_f1(tp, fn, removed)
            axes[1].plot(rounds, f1, linewidth=1.5, label=label)

    axes[0].set_title('Test Accuracy')
    axes[0].set_xlabel('Evaluation Round')
    axes[0].legend(fontsize=9)

    axes[1].set_title('F1 de Deteccao por Round')
    axes[1].set_xlabel('Round')
    axes[1].legend(fontsize=9)

    plt.suptitle('Comparacao entre Experimentos', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
elif len(agentlog_files) == 1:
    print('Apenas 1 arquivo agentlog disponivel. Rode mais experimentos para comparar.')
else:
    print('Nenhum arquivo agentlog encontrado.')

In [ ]:
defense_by_cc = {}

for path in agentlog_files:
    d = load_agentlog(path)
    cc = d['_config']['cc']
    rounds, removed, fpr, frr, tp, fn = detection_metrics(d)
    defense_by_cc.setdefault(cc, []).append((rounds, fpr, frr))


def aggregate(series_list):
    """Media de FPR/FRR por round, agregando os experimentos de um mesmo cc."""
    by_round = {}
    for rounds, fpr, frr in series_list:
        for r, a, b in zip(rounds, fpr, frr):
            by_round.setdefault(int(r), []).append((a, b))
    rs = sorted(by_round)
    return (np.array(rs),
            np.array([np.mean([v[0] for v in by_round[r]]) for r in rs]),
            np.array([np.mean([v[1] for v in by_round[r]]) for r in rs]))


if len(defense_by_cc) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))

    for cc, series_list in sorted(defense_by_cc.items()):
        rounds, fpr, frr = aggregate(series_list)
        axes[0].plot(rounds, fpr, linewidth=2, label=f'cc={cc}')
        axes[1].plot(rounds, frr, linewidth=2, label=f'cc={cc}')

    axes[0].set_title('FPR por Rodada (media agregada por CC)')
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('FPR')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].set_title('FRR por Rodada (media agregada por CC)')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('FRR')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('FPR e FRR por Configuracao de CC', fontsize=14, fontweight='bold')
    plt.tight_layout()
    os.makedirs(PLOTS_DIR, exist_ok=True)
    plt.savefig(os.path.join(PLOTS_DIR, 'defense_fpr_frr_by_cc.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Plot salvo: {os.path.join(PLOTS_DIR, "defense_fpr_frr_by_cc.png")}')
else:
    print(f'Dados insuficientes para comparar CCs (encontrados {len(defense_by_cc)} valores distintos).')